# Data Analysis of the filtered Materials dataset

---

In [ ]:
import duckdb

from src.data_preprocessing.config import INPUT_FILE_MATERIALS_1w_enc_f

In [ ]:
con = duckdb.connect()

In [ ]:
# Attach the parquet file as a table
con.execute(f"CREATE OR REPLACE TABLE materials AS SELECT * FROM '{INPUT_FILE_MATERIALS_1w_enc_f}';")

### Missing percentage per column

In [ ]:
print("Null percentage per column:")

# Get list of all column names using DESCRIBE
columns = con.execute(f"DESCRIBE SELECT * FROM '{INPUT_FILE_MATERIALS_1w_enc_f}'").fetchdf()['column_name'].tolist()

results = []
total_rows = con.execute(f"SELECT COUNT(*) FROM '{INPUT_FILE_MATERIALS_1w_enc_f}'").fetchone()[0]

for col in columns:
    null_count = con.execute(f"SELECT COUNT(*) - COUNT({col}) FROM '{INPUT_FILE_MATERIALS_1w_enc_f}'").fetchone()[0]
    null_percentage = round(100.0 * null_count / total_rows, 2)
    results.append((col, null_percentage))

# Sort and print
results.sort(key=lambda x: x[1], reverse=True)
for col, perc in results:
    print(f"{col}: {perc:.2f}% nulls")

### Unique value count per column (Cardinality)

In [ ]:
# Get column names
columns = con.execute("SELECT name FROM pragma_table_info('materials');").fetchdf()['name'].tolist()

# Calculate unique counts per column
results = []
for col in columns:
    unique_count = con.execute(f"SELECT COUNT(DISTINCT {col}) FROM materials;").fetchone()[0]
    results.append((col, unique_count))

# Print nicely
import pandas as pd

unique_counts = pd.DataFrame(results, columns=['column_name', 'unique_values'])
print("\n=== Unique Values per Column ===")
print(unique_counts)

categorical_cols = con.execute("""
                               SELECT name
                               FROM pragma_table_info('materials')
                               WHERE type IN ('VARCHAR', 'STRING', 'TEXT');
                               """).fetchdf()['name'].tolist()

for col in categorical_cols:
    print(f"\n=== Top 10 Frequent Values for '{col}' ===")
    result = con.execute(f"""
        SELECT {col} AS value, COUNT(*) AS freq
        FROM materials
        GROUP BY {col}
        ORDER BY freq DESC
        LIMIT 10;
    """).fetchdf()
    print(result)

### Dataset summary: number of rows, memory usage (approximate)

In [ ]:
# Get total number of rows
summary = con.execute("""
                      SELECT COUNT(*) AS total_rows
                      FROM materials;
                      """).fetchdf()

print("\n=== Dataset Summary ===")
print(summary)

# Approximate size: Sum LENGTH for string columns
string_cols = con.execute("""
                          SELECT name
                          FROM pragma_table_info('materials')
                          WHERE type IN ('VARCHAR', 'STRING', 'TEXT');
                          """).fetchdf()['name'].tolist()

if string_cols:
    length_sum_expr = " + ".join([f"LENGTH({col})" for col in string_cols])
    size_query = f"""
        SELECT ROUND(SUM({length_sum_expr}) / 1024 / 1024, 2) AS approx_string_data_MB
        FROM materials;
    """
    approx_size = con.execute(size_query).fetchdf()
    print("\n=== Approximate Size of String Data (MB) ===")
    print(approx_size)
else:
    print("\nNo string columns to estimate size.")

# Schema info
schema = con.execute("""
                     SELECT name AS column_name, type AS data_type
                     FROM pragma_table_info('materials');
                     """).fetchdf()
print("\n=== Schema Information ===")
print(schema)

In [ ]:
con.close()